In [1]:
import os
import pandas as pd
import psycopg2
from dotenv import load_dotenv

load_dotenv()


def get_connection():
    return psycopg2.connect(
        host=os.getenv("PGHOST"),
        database=os.getenv("PGDATABASE"),
        user=os.getenv("PGUSER"),
        password=os.getenv("PGPASSWORD"),
        sslmode=os.getenv("PGSSLMODE")
    )


def execute_query(query):
    conn = None
    cur = None

    try:
        conn = get_connection()
        cur = conn.cursor()

        cur.execute(query)
        conn.commit()

        print("✓ Sweet! Data updated")

    except Exception as e:
        if conn:
            conn.rollback()

        print("Errore:", e)

    finally:
        if cur:
            cur.close()

        if conn:
            conn.close()

In [4]:
## query per PREZZI STRUMENTI
#
query = """
WITH normalized AS (
    SELECT
        r.raw_price_id,

        -- Identificativi normalizzati
        NULLIF(UPPER(TRIM(r.ticker)), '') AS ticker,
        NULLIF(UPPER(TRIM(r.isin)), '') AS isin,

        -- Data
        r.price_date,

        -- Prezzi
        r.open_price,
        r.high_price,
        r.low_price,
        r.close_price,
        r.adjusted_close,

        -- Volume
        r.volume,

        -- Valuta normalizzata
        NULLIF(UPPER(TRIM(r.currency)), '') AS currency,

        -- Fonte
        NULLIF(TRIM(r.source_name), '') AS source_name,

        -- Timestamp di caricamento RAW
        r.loaded_at,

        -- Identificazione dei duplicati
        ROW_NUMBER() OVER (
            PARTITION BY
                COALESCE(
                    NULLIF(UPPER(TRIM(r.isin)), ''),
                    'TICKER:' || NULLIF(UPPER(TRIM(r.ticker)), '')
                ),
                r.price_date,
                NULLIF(TRIM(r.source_name), '')
            ORDER BY r.raw_price_id DESC
        ) AS rn

    FROM raw.instrument_prices r
),

valid_data AS (
    SELECT *
    FROM normalized

    WHERE rn = 1

    -- Deve esserci almeno un identificativo
    AND (
        ticker IS NOT NULL
        OR isin IS NOT NULL
    )

    -- Data obbligatoria
    AND price_date IS NOT NULL

    -- Almeno un prezzo utilizzabile
    AND (
        close_price IS NOT NULL
        OR adjusted_close IS NOT NULL
    )

    -- Prezzi non negativi
    AND (open_price IS NULL OR open_price >= 0)
    AND (high_price IS NULL OR high_price >= 0)
    AND (low_price IS NULL OR low_price >= 0)
    AND (close_price IS NULL OR close_price >= 0)
    AND (adjusted_close IS NULL OR adjusted_close >= 0)

    -- Volume non negativo
    AND (volume IS NULL OR volume >= 0)

    -- Controlli di coerenza OHLC
    AND (
        high_price IS NULL
        OR low_price IS NULL
        OR high_price >= low_price
    )

    AND (
        high_price IS NULL
        OR open_price IS NULL
        OR high_price >= open_price
    )

    AND (
        high_price IS NULL
        OR close_price IS NULL
        OR high_price >= close_price
    )

    AND (
        low_price IS NULL
        OR open_price IS NULL
        OR low_price <= open_price
    )

    AND (
        low_price IS NULL
        OR close_price IS NULL
        OR low_price <= close_price
    )
)

INSERT INTO staging.instrument_prices (
    raw_price_id,
    ticker,
    isin,
    price_date,
    open_price,
    high_price,
    low_price,
    close_price,
    adjusted_close,
    volume,
    currency,
    source_name,
    raw_loaded_at
)

SELECT
    v.raw_price_id,
    v.ticker,
    v.isin,
    v.price_date,
    v.open_price,
    v.high_price,
    v.low_price,
    v.close_price,
    v.adjusted_close,
    v.volume,
    v.currency,
    v.source_name,
    v.loaded_at

FROM valid_data v

WHERE NOT EXISTS (
    SELECT 1
    FROM staging.instrument_prices s
    WHERE s.raw_price_id = v.raw_price_id
);
"""

execute_query(query)

✓ Sweet! Data updated


In [5]:
# FOREX RATES

query = """
WITH normalized AS (
    SELECT
        r.raw_fx_rate_id,

        -- Data
        r.rate_date,

        -- Valute normalizzate
        NULLIF(UPPER(TRIM(r.base_currency)), '') AS base_currency,
        NULLIF(UPPER(TRIM(r.quote_currency)), '') AS quote_currency,

        -- Tasso FX
        r.fx_rate,

        -- Tipo tasso normalizzato
        COALESCE(
            NULLIF(UPPER(TRIM(r.rate_type)), ''),
            'CLOSE'
        ) AS rate_type,

        -- Fonte
        NULLIF(TRIM(r.source_name), '') AS source_name,

        -- Lineage
        r.ingestion_file_id,
        r.load_at,

        -- Identificazione duplicati
        ROW_NUMBER() OVER (
            PARTITION BY
                r.rate_date,
                NULLIF(UPPER(TRIM(r.base_currency)), ''),
                NULLIF(UPPER(TRIM(r.quote_currency)), ''),
                COALESCE(
                    NULLIF(UPPER(TRIM(r.rate_type)), ''),
                    'CLOSE'
                ),
                NULLIF(TRIM(r.source_name), '')
            ORDER BY
                r.load_at DESC,
                r.raw_fx_rate_id DESC
        ) AS rn

    FROM raw.fx_rates r
),

valid_data AS (
    SELECT *
    FROM normalized

    WHERE rn = 1

    -- Data obbligatoria
    AND rate_date IS NOT NULL

    -- Valute obbligatorie
    AND base_currency IS NOT NULL
    AND quote_currency IS NOT NULL

    -- Codice valuta ISO a 3 caratteri
    AND LENGTH(base_currency) = 3
    AND LENGTH(quote_currency) = 3

    -- Evita coppie EUR/EUR, USD/USD, ecc.
    AND base_currency <> quote_currency

    -- FX rate deve essere positivo
    AND fx_rate IS NOT NULL
    AND fx_rate > 0

    -- Tipo tasso obbligatorio dopo normalizzazione
    AND rate_type IS NOT NULL
)

INSERT INTO staging.fx_rates (
    raw_fx_rate_id,
    rate_date,
    base_currency,
    quote_currency,
    fx_rate,
    rate_type,
    source_name,
    ingestion_file_id,
    raw_loaded_at
)

SELECT
    v.raw_fx_rate_id,
    v.rate_date,
    v.base_currency,
    v.quote_currency,
    v.fx_rate,
    v.rate_type,
    v.source_name,
    v.ingestion_file_id,
    v.load_at

FROM valid_data v

-- Evita di ricaricare la stessa riga RAW
WHERE NOT EXISTS (
    SELECT 1
    FROM staging.fx_rates s
    WHERE s.raw_fx_rate_id = v.raw_fx_rate_id
);
"""

execute_query(query)



✓ Sweet! Data updated
